# 第15章 硬件与虚拟机 - 处理器与并行处理
# Chapter 15 Hardware & Virtual Machines - Processors & Parallel Processing

---

**Cambridge A-Level Computer Science (9618) - A2 Level Paper 3**

本笔记本涵盖以下内容：
- RISC 与 CISC 处理器架构比较
- 流水线 (Pipelining) 技术：阶段、冒险、分支预测
- 并行处理 (Parallel Processing)：SIMD、MIMD
- 多核处理器 (Multi-core Processors) 及其优势
- GPU 计算基础
- 阿姆达尔定律 (Amdahl's Law) 与加速比计算
- 交互式演示与可视化

In [ ]:
# ============================================================
# 环境配置 - 每个笔记本开头必须运行
# ============================================================
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import numpy as np
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets

# 中文字体配置
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['figure.dpi'] = 100

print("✅ 环境配置完成！")

---
## 1. RISC 与 CISC 架构比较

### 1.1 什么是指令集架构 (Instruction Set Architecture, ISA)？

**指令集架构**是处理器能够理解和执行的所有指令的集合。它定义了：
- 处理器能执行哪些操作（加法、移动数据、跳转等）
- 指令的格式（操作码 + 操作数）
- 寻址模式（如何找到数据）
- 寄存器的数量和用途

现代处理器主要分为两种架构：**RISC** 和 **CISC**。

### 1.2 CISC (Complex Instruction Set Computer) - 复杂指令集计算机

**核心理念**：用尽量少的指令完成复杂任务（减少程序员/编译器的工作量）。

**特点**：
- 指令数量多（通常 200+ 条指令）
- 指令长度可变（Variable-length instructions）
- 单条指令可以执行复杂操作（如直接从内存读取、计算、写回）
- 指令执行需要多个时钟周期（Multi-cycle execution）
- 寻址模式多样
- 通用寄存器较少

**代表**：Intel x86, AMD x86-64

**举例**：一条 CISC 指令可以完成：
```
MULT [address1], [address2]   ; 从内存取两个数，相乘，结果存回内存
```

### 1.3 RISC (Reduced Instruction Set Computer) - 精简指令集计算机

**核心理念**：用简单的指令高效执行（让硬件更快，每条指令一个时钟周期）。

**特点**：
- 指令数量少（通常 50-100 条）
- 指令长度固定（Fixed-length instructions）
- 每条指令只执行一个简单操作
- 大多数指令一个时钟周期完成（Single-cycle execution）
- Load/Store 架构：只有 LOAD 和 STORE 指令访问内存，其他指令只操作寄存器
- 通用寄存器数量多
- 非常适合流水线 (pipelining)

**代表**：ARM, MIPS, RISC-V

**举例**：同样的乘法操作需要多条 RISC 指令：
```
LOAD R1, [address1]    ; 从内存加载到寄存器
LOAD R2, [address2]    ; 从内存加载到寄存器  
MULT R3, R1, R2        ; 寄存器之间相乘
STORE R3, [address3]   ; 结果存回内存
```

In [ ]:
# ============================================================
# RISC vs CISC 对比表格可视化
# ============================================================

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('RISC vs CISC 架构对比', fontsize=18, fontweight='bold', pad=20)

# 表头
headers = ['特性 / Feature', 'CISC', 'RISC']
header_x = [1.5, 6, 11]
for i, (h, x) in enumerate(zip(headers, header_x)):
    color = '#2c3e50' if i == 0 else ('#e74c3c' if i == 1 else '#27ae60')
    ax.add_patch(FancyBboxPatch((x-2.2, 9.0), 4.2, 0.7, boxstyle='round,pad=0.1',
                                facecolor=color, edgecolor='white', linewidth=2))
    ax.text(x, 9.35, h, ha='center', va='center', fontsize=13, fontweight='bold', color='white')

# 数据行
rows = [
    ['指令数量', '200+ (多)', '50-100 (少)'],
    ['指令长度', '可变长度', '固定长度'],
    ['指令复杂度', '复杂（多步操作）', '简单（单步操作）'],
    ['执行周期', '多个时钟周期', '通常 1 个周期'],
    ['内存访问', '任何指令可访问', '仅 Load/Store'],
    ['寄存器数量', '较少 (8-16)', '较多 (32+)'],
    ['流水线友好', '困难', '非常适合'],
    ['代码密度', '高（程序短）', '低（程序长）'],
    ['功耗', '较高', '较低'],
    ['代表芯片', 'Intel x86', 'ARM, RISC-V'],
]

for j, (feat, cisc, risc) in enumerate(rows):
    y = 8.2 - j * 0.8
    bg = '#f8f9fa' if j % 2 == 0 else '#ffffff'
    for x_start in [-0.7, 3.8, 8.8]:
        ax.add_patch(FancyBboxPatch((x_start, y-0.3), 4.2, 0.65, boxstyle='round,pad=0.05',
                                    facecolor=bg, edgecolor='#dee2e6', linewidth=0.5))
    ax.text(1.5, y+0.02, feat, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(6, y+0.02, cisc, ha='center', va='center', fontsize=10, color='#c0392b')
    ax.text(11, y+0.02, risc, ha='center', va='center', fontsize=10, color='#229954')

plt.tight_layout()
plt.show()

### 1.4 RISC 与 CISC 的考试要点

**考试中经常出现的问题**：

| 问题类型 | 关键回答点 |
|---------|----------|
| 为什么 RISC 更适合流水线？ | 因为所有指令长度相同、执行时间一致，流水线各阶段时间相等 |
| 为什么 CISC 代码更短？ | 一条复杂指令 = 多条简单指令的功能 |
| 为什么手机用 ARM (RISC)？ | RISC 指令简单 → 芯片面积小 → 功耗低 → 省电 |
| 两者的现代融合趋势？ | 现代 CISC (如 x86) 内部将复杂指令拆分为 micro-ops（类似 RISC） |

> **考试技巧**：对比题中，务必从两个方面作答，不要只说一方。每个观点都给出「特性 → 结果」的因果链。

---
## 2. 流水线技术 (Pipelining)

### 2.1 什么是流水线？

**流水线 (Pipelining)** 是一种让处理器同时执行多条指令的不同阶段的技术，就像工厂的装配线。

**类比**：想象一个洗衣服的过程：
1. 洗涤 (Wash) - 30分钟
2. 烘干 (Dry) - 30分钟  
3. 折叠 (Fold) - 30分钟

不用流水线：洗完一批衣服（90分钟）才开始下一批。
用流水线：第一批进入烘干机后，第二批立即进入洗衣机！

### 2.2 指令流水线的五个阶段

经典的 **五阶段流水线 (5-Stage Pipeline)** 包括：

| 阶段 | 英文 | 缩写 | 功能 |
|------|------|------|------|
| 取指 | Fetch | IF | 从内存中取出指令 |
| 译码 | Decode | ID | 解析指令的操作码和操作数 |
| 执行 | Execute | EX | ALU 执行运算 |
| 访存 | Memory Access | MEM | 读写内存（如需要） |
| 写回 | Write Back | WB | 将结果写回寄存器 |

In [ ]:
# ============================================================
# 流水线阶段可视化：无流水线 vs 有流水线
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

stages = ['IF\n取指', 'ID\n译码', 'EX\n执行', 'MEM\n访存', 'WB\n写回']
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
n_instructions = 4
n_stages = 5

# ---- 上图：无流水线（顺序执行）----
ax = axes[0]
ax.set_title('无流水线（Sequential Execution）- 每条指令完成后才开始下一条', fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, n_instructions * n_stages + 0.5)
ax.set_ylim(-0.5, n_instructions + 0.5)
ax.set_xlabel('时钟周期 (Clock Cycle)', fontsize=12)
ax.set_ylabel('指令 (Instruction)', fontsize=12)
ax.set_yticks(range(n_instructions))
ax.set_yticklabels([f'指令 {i+1}' for i in range(n_instructions)])

for i in range(n_instructions):
    for s in range(n_stages):
        x = i * n_stages + s
        rect = plt.Rectangle((x, i - 0.35), 0.9, 0.7, facecolor=colors[s], edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + 0.45, i, stages[s], ha='center', va='center', fontsize=7, fontweight='bold', color='white')

total_seq = n_instructions * n_stages
ax.set_xticks(range(total_seq))
ax.set_xticklabels([str(i+1) for i in range(total_seq)], fontsize=8)
ax.text(total_seq/2, n_instructions + 0.2, f'总共需要 {total_seq} 个时钟周期', ha='center', fontsize=12, color='red', fontweight='bold')

# ---- 下图：有流水线 ----
ax = axes[1]
ax.set_title('有流水线（Pipelined Execution）- 各阶段重叠执行', fontsize=14, fontweight='bold')
total_pipe = n_stages + n_instructions - 1
ax.set_xlim(-0.5, total_pipe + 0.5)
ax.set_ylim(-0.5, n_instructions + 0.5)
ax.set_xlabel('时钟周期 (Clock Cycle)', fontsize=12)
ax.set_ylabel('指令 (Instruction)', fontsize=12)
ax.set_yticks(range(n_instructions))
ax.set_yticklabels([f'指令 {i+1}' for i in range(n_instructions)])

for i in range(n_instructions):
    for s in range(n_stages):
        x = i + s  # 关键：每条指令错开一个周期开始
        rect = plt.Rectangle((x, i - 0.35), 0.9, 0.7, facecolor=colors[s], edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + 0.45, i, stages[s], ha='center', va='center', fontsize=7, fontweight='bold', color='white')

ax.set_xticks(range(total_pipe))
ax.set_xticklabels([str(i+1) for i in range(total_pipe)], fontsize=8)
ax.text(total_pipe/2, n_instructions + 0.2, f'总共需要 {total_pipe} 个时钟周期', ha='center', fontsize=12, color='green', fontweight='bold')

# 图例
legend_patches = [mpatches.Patch(color=c, label=s.replace('\n', ' ')) for c, s in zip(colors, stages)]
fig.legend(handles=legend_patches, loc='lower center', ncol=5, fontsize=10, frameon=True)

plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

speedup = total_seq / total_pipe
print(f"\n📊 性能对比：")
print(f"   无流水线：{total_seq} 个周期")
print(f"   有流水线：{total_pipe} 个周期")
print(f"   加速比 (Speedup)：{speedup:.2f}x")
print(f"\n💡 理论上，{n_stages} 阶段流水线在处理大量指令时，加速比趋近于 {n_stages}x")

### 2.3 流水线加速比公式

对于 $n$ 条指令、$k$ 个流水线阶段：

$$\text{无流水线时间} = n \times k \text{ 个周期}$$

$$\text{有流水线时间} = k + (n - 1) \text{ 个周期}$$

$$\text{加速比} = \frac{n \times k}{k + (n - 1)}$$

当 $n \to \infty$ 时：

$$\text{加速比} \to k$$

即流水线的最大加速比等于流水线的阶段数！

In [ ]:
# ============================================================
# 交互式流水线加速比计算器
# ============================================================

def pipeline_speedup_demo(n_instructions=20, n_stages=5):
    """计算并可视化流水线加速比"""
    # 计算不同指令数的加速比
    instructions = np.arange(1, n_instructions + 1)
    seq_time = instructions * n_stages
    pipe_time = n_stages + (instructions - 1)
    speedup = seq_time / pipe_time
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：执行时间对比
    ax1.plot(instructions, seq_time, 'r-o', label='无流水线', markersize=4)
    ax1.plot(instructions, pipe_time, 'g-o', label='有流水线', markersize=4)
    ax1.set_xlabel('指令数量', fontsize=12)
    ax1.set_ylabel('时钟周期数', fontsize=12)
    ax1.set_title(f'{n_stages} 阶段流水线：执行时间对比', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.fill_between(instructions, pipe_time, seq_time, alpha=0.15, color='green', label='节省的时间')
    
    # 右图：加速比
    ax2.plot(instructions, speedup, 'b-o', markersize=4, linewidth=2)
    ax2.axhline(y=n_stages, color='red', linestyle='--', alpha=0.7, label=f'理论最大加速比 = {n_stages}')
    ax2.set_xlabel('指令数量', fontsize=12)
    ax2.set_ylabel('加速比 (Speedup)', fontsize=12)
    ax2.set_title('流水线加速比', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, n_stages + 1)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 {n_stages} 阶段流水线，执行 {n_instructions} 条指令：")
    print(f"   加速比 = ({n_instructions} × {n_stages}) / ({n_stages} + {n_instructions} - 1)")
    print(f"         = {n_instructions * n_stages} / {n_stages + n_instructions - 1}")
    print(f"         = {n_instructions * n_stages / (n_stages + n_instructions - 1):.3f}")

# 创建交互控件
widgets.interact(
    pipeline_speedup_demo,
    n_instructions=widgets.IntSlider(min=1, max=100, step=1, value=20, description='指令数:'),
    n_stages=widgets.IntSlider(min=2, max=10, step=1, value=5, description='阶段数:')
)

### 2.4 流水线冒险 (Pipeline Hazards)

流水线并非总能完美运行，三种**冒险 (Hazard)** 会导致流水线暂停（pipeline stall / bubble）：

#### (1) 数据冒险 (Data Hazard)
**原因**：后一条指令依赖前一条指令的结果，但结果还没算完。

```
ADD R1, R2, R3    ; R1 = R2 + R3  （正在执行阶段）
SUB R4, R1, R5    ; R4 = R1 - R5  （需要 R1 的值，但还没写回！）
```

**解决方案**：
- **数据前递 (Data Forwarding / Bypassing)**：在结果写回寄存器之前，直接将结果传递给需要的阶段
- **插入气泡 (Bubble / NOP)**：暂停流水线等待数据
- **指令重排 (Instruction Reordering)**：编译器调整指令顺序，避免依赖

#### (2) 控制冒险 (Control Hazard / Branch Hazard)
**原因**：条件跳转指令（如 `BEQ`, `JMP`）改变了程序执行流，已经取入流水线的指令可能是错误的。

```
BEQ R1, R2, LABEL  ; 如果 R1==R2，跳到 LABEL（但结果还没出来）
ADD R3, R4, R5      ; 已经被取入流水线——如果跳转了，这条就不该执行！
SUB R6, R7, R8      ; 同上
```

**解决方案**：
- **分支预测 (Branch Prediction)**：猜测跳转是否发生，猜对继续，猜错清空流水线 (flush)
- **延迟槽 (Delayed Branch)**：在跳转后放一条无关指令，利用等待时间

#### (3) 结构冒险 (Structural Hazard)
**原因**：两条指令同时需要使用同一个硬件资源（如同时读写内存）。

**解决方案**：
- 增加硬件资源（如分离指令缓存和数据缓存 — Harvard Architecture）
- 插入气泡等待

In [ ]:
# ============================================================
# 流水线冒险可视化：数据冒险与气泡
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

stages_short = ['IF', 'ID', 'EX', 'MEM', 'WB']
colors_stage = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
bubble_color = '#bdc3c7'

# ---- 上图：有数据冒险（需要气泡）----
ax = axes[0]
ax.set_title('数据冒险 (Data Hazard) - 需要插入气泡 (Bubble/Stall)', fontsize=13, fontweight='bold')

instructions = [
    'ADD R1,R2,R3',
    '--- 气泡 ---',
    '--- 气泡 ---',
    'SUB R4,R1,R5',
    'AND R6,R7,R8',
]

# ADD 正常执行
pipeline_data = [
    [0,1,2,3,4],         # ADD: 正常 IF ID EX MEM WB
    [1,-1,-1,-1,-1],     # Bubble
    [2,-1,-1,-1,-1],     # Bubble
    [3,4,5,6,7],         # SUB: 延迟2个周期
    [4,5,6,7,8],         # AND: 延迟2个周期
]

for i, (instr, data) in enumerate(zip(instructions, pipeline_data)):
    for s, cycle in enumerate(data):
        if cycle == -1:
            continue
        is_bubble = '气泡' in instr
        c = bubble_color if is_bubble else colors_stage[s]
        label = 'STALL' if is_bubble else stages_short[s]
        rect = plt.Rectangle((cycle, i - 0.35), 0.9, 0.7, facecolor=c, edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(cycle + 0.45, i, label, ha='center', va='center', fontsize=8, fontweight='bold',
                color='white' if not is_bubble else '#7f8c8d')

ax.set_xlim(-0.5, 10)
ax.set_ylim(-0.5, 5.5)
ax.set_yticks(range(5))
ax.set_yticklabels(instructions, fontsize=9, fontfamily='monospace')
ax.set_xlabel('时钟周期', fontsize=11)
ax.set_xticks(range(10))
ax.set_xticklabels([str(i+1) for i in range(10)])

# 画箭头标注依赖
ax.annotate('R1 依赖！', xy=(4, 0.35), xytext=(5.5, 1.5),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

# ---- 下图：用数据前递解决 ----
ax = axes[1]
ax.set_title('数据前递 (Data Forwarding) - 无需气泡', fontsize=13, fontweight='bold')

instructions2 = ['ADD R1,R2,R3', 'SUB R4,R1,R5', 'AND R6,R7,R8', 'OR R9,R10,R11']
for i in range(4):
    for s in range(5):
        x = i + s
        rect = plt.Rectangle((x, i - 0.35), 0.9, 0.7, facecolor=colors_stage[s], edgecolor='white', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x + 0.45, i, stages_short[s], ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# 画前递箭头
ax.annotate('', xy=(3, 1 - 0.1), xytext=(2.9, 0 + 0.1),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2.5, linestyle='--'))
ax.text(3.5, 0.5, '前递\nForwarding', fontsize=9, color='red', fontweight='bold', ha='center')

ax.set_xlim(-0.5, 9)
ax.set_ylim(-0.5, 4.5)
ax.set_yticks(range(4))
ax.set_yticklabels(instructions2, fontsize=9, fontfamily='monospace')
ax.set_xlabel('时钟周期', fontsize=11)
ax.set_xticks(range(9))
ax.set_xticklabels([str(i+1) for i in range(9)])

plt.tight_layout()
plt.show()

### 2.5 分支预测 (Branch Prediction)

**分支预测**是处理器用来猜测条件跳转结果的技术，以减少控制冒险造成的流水线清空。

#### 静态分支预测 (Static Branch Prediction)
- **总是预测不跳转 (Predict Not Taken)**：假设条件不成立，继续执行下一条
- **总是预测跳转 (Predict Taken)**：假设条件成立
- **向后跳转预测为 taken**：循环通常向后跳转（回到循环头）

#### 动态分支预测 (Dynamic Branch Prediction)
- **1-bit 预测器**：记录上次是否跳转，下次做相同预测
- **2-bit 预测器 (饱和计数器)**：需要连续两次预测错误才改变预测方向
  - 状态：强不跳转 → 弱不跳转 → 弱跳转 → 强跳转
  - 对循环非常有效（只在进入和退出时错误）

#### 预测错误的代价
- 需要**清空流水线 (Pipeline Flush)**
- 所有错误取入的指令作废
- 流水线越深（阶段越多），清空的代价越大

In [ ]:
# ============================================================
# 2-bit 分支预测器状态机可视化
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 7)
ax.axis('off')
ax.set_title('2-bit 饱和计数器分支预测器 (2-bit Saturating Counter)', fontsize=15, fontweight='bold')

# 四个状态
states = [
    (1.5, 3, '00\n强不跳转\nStrongly\nNot Taken', '#27ae60'),
    (4, 3, '01\n弱不跳转\nWeakly\nNot Taken', '#82e0aa'),
    (6.5, 3, '10\n弱跳转\nWeakly\nTaken', '#f5b041'),
    (9, 3, '11\n强跳转\nStrongly\nTaken', '#e74c3c'),
]

for x, y, label, color in states:
    circle = plt.Circle((x, y), 1.1, facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')

# 转移箭头 - 实际跳转 (Taken)
arrow_style = dict(arrowstyle='->', color='#e74c3c', lw=2)
# 00 -> 01 (Taken)
ax.annotate('', xy=(2.95, 3.6), xytext=(2.55, 3.6), arrowprops=arrow_style)
ax.text(2.75, 4.5, 'T', fontsize=10, color='red', fontweight='bold', ha='center')
# 01 -> 10 (Taken)
ax.annotate('', xy=(5.45, 3.6), xytext=(5.05, 3.6), arrowprops=arrow_style)
ax.text(5.25, 4.5, 'T', fontsize=10, color='red', fontweight='bold', ha='center')
# 10 -> 11 (Taken)
ax.annotate('', xy=(7.95, 3.6), xytext=(7.55, 3.6), arrowprops=arrow_style)
ax.text(7.75, 4.5, 'T', fontsize=10, color='red', fontweight='bold', ha='center')

# 转移箭头 - 实际不跳转 (Not Taken)
arrow_style2 = dict(arrowstyle='->', color='#27ae60', lw=2)
# 01 -> 00 (NT)
ax.annotate('', xy=(2.55, 2.4), xytext=(2.95, 2.4), arrowprops=arrow_style2)
ax.text(2.75, 1.3, 'NT', fontsize=10, color='green', fontweight='bold', ha='center')
# 10 -> 01 (NT)
ax.annotate('', xy=(5.05, 2.4), xytext=(5.45, 2.4), arrowprops=arrow_style2)
ax.text(5.25, 1.3, 'NT', fontsize=10, color='green', fontweight='bold', ha='center')
# 11 -> 10 (NT)
ax.annotate('', xy=(7.55, 2.4), xytext=(7.95, 2.4), arrowprops=arrow_style2)
ax.text(7.75, 1.3, 'NT', fontsize=10, color='green', fontweight='bold', ha='center')

# 自循环
ax.annotate('T', xy=(9, 4.15), fontsize=10, color='red', fontweight='bold', ha='center')
ax.annotate('NT', xy=(1.5, 1.65), fontsize=10, color='green', fontweight='bold', ha='center')

# 说明
ax.text(5.25, 6.3, '预测规则：状态为 00 或 01 → 预测不跳转（Not Taken）', fontsize=11, ha='center', fontweight='bold')
ax.text(5.25, 5.7, '预测规则：状态为 10 或 11 → 预测跳转（Taken）', fontsize=11, ha='center', fontweight='bold')
ax.text(5.25, -0.3, 'T = 实际跳转 (Taken)   NT = 实际不跳转 (Not Taken)', fontsize=11, ha='center', style='italic')

plt.tight_layout()
plt.show()

---
## 3. 并行处理 (Parallel Processing)

### 3.1 弗林分类法 (Flynn's Taxonomy)

弗林分类法根据**指令流 (Instruction Stream)** 和**数据流 (Data Stream)** 将计算机架构分为四类：

| 类别 | 全称 | 指令流 | 数据流 | 描述 |
|------|------|--------|--------|------|
| **SISD** | Single Instruction, Single Data | 1 | 1 | 传统单处理器，一次处理一条指令、一个数据 |
| **SIMD** | Single Instruction, Multiple Data | 1 | 多 | 一条指令同时处理多个数据（向量运算） |
| **MISD** | Multiple Instruction, Single Data | 多 | 1 | 多条指令处理同一数据（理论，很少使用） |
| **MIMD** | Multiple Instruction, Multiple Data | 多 | 多 | 多个处理器独立执行不同指令和数据 |

### 3.2 SIMD (Single Instruction, Multiple Data)

**核心思想**：同一条指令同时对多个数据执行相同的操作。

**应用场景**：
- **图像处理**：对每个像素执行相同的亮度调整
- **音频处理**：对每个音频采样执行相同的滤波
- **科学计算**：大规模向量/矩阵运算
- **GPU 计算**：GPU 本质上是大规模 SIMD 处理器

**举例**：将数组中每个元素乘以2
```
SISD：逐个处理 A[0]*2, A[1]*2, A[2]*2, A[3]*2（4步）
SIMD：一次处理 A[0:3]*2（1步，4个数据并行）
```

### 3.3 MIMD (Multiple Instruction, Multiple Data)

**核心思想**：多个处理器各自执行不同的指令，处理不同的数据。

**分类**：
- **共享内存 (Shared Memory)**：所有处理器共享同一内存空间（如多核 CPU）
- **分布式内存 (Distributed Memory)**：每个处理器有自己的内存，通过网络通信（如计算集群）

**应用场景**：
- 多核 CPU 执行多任务
- 服务器集群处理不同用户请求
- 分布式计算（如 MapReduce）

In [ ]:
# ============================================================
# Flynn's Taxonomy 可视化
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("弗林分类法 (Flynn's Taxonomy)", fontsize=16, fontweight='bold')

configs = [
    ('SISD\nSingle Instruction, Single Data', axes[0, 0], 1, 1, '#3498db'),
    ('SIMD\nSingle Instruction, Multiple Data', axes[0, 1], 1, 4, '#e74c3c'),
    ('MISD\nMultiple Instruction, Single Data', axes[1, 0], 4, 1, '#f39c12'),
    ('MIMD\nMultiple Instruction, Multiple Data', axes[1, 1], 4, 4, '#2ecc71'),
]

for title, ax, n_instr, n_data, color in configs:
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')
    
    # 指令流
    instr_labels = [f'I{i+1}' if n_instr > 1 else 'I' for i in range(n_instr)]
    for i, lbl in enumerate(instr_labels):
        y = 5 - i * (3.0 / max(n_instr - 1, 1)) if n_instr > 1 else 3.5
        ax.add_patch(FancyBboxPatch((0.5, y - 0.3), 1.8, 0.6, boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white', alpha=0.8))
        ax.text(1.4, y, lbl, ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # 处理单元
    ax.add_patch(FancyBboxPatch((3.5, 2), 2.5, 2, boxstyle='round,pad=0.15',
                                facecolor='#ecf0f1', edgecolor=color, linewidth=2))
    ax.text(4.75, 3, 'PU', ha='center', va='center', fontsize=14, fontweight='bold', color=color)
    
    # 数据流
    data_labels = [f'D{i+1}' if n_data > 1 else 'D' for i in range(n_data)]
    for i, lbl in enumerate(data_labels):
        y = 5 - i * (3.0 / max(n_data - 1, 1)) if n_data > 1 else 3.5
        ax.add_patch(FancyBboxPatch((7.2, y - 0.3), 1.8, 0.6, boxstyle='round,pad=0.1',
                                    facecolor='#95a5a6', edgecolor='white', alpha=0.8))
        ax.text(8.1, y, lbl, ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # 箭头
    ax.annotate('', xy=(3.5, 3), xytext=(2.3, 3), arrowprops=dict(arrowstyle='->', lw=2, color=color))
    ax.annotate('', xy=(7.2, 3), xytext=(6.0, 3), arrowprops=dict(arrowstyle='->', lw=2, color='#95a5a6'))

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SIMD vs SISD 执行对比演示
# ============================================================

import time

# 模拟 SISD（标量）处理
def sisd_add(a, b):
    """逐个元素相加 - 模拟 SISD"""
    result = []
    for i in range(len(a)):
        result.append(a[i] + b[i])
    return result

# 模拟 SIMD（向量）处理
def simd_add(a, b):
    """使用 NumPy 向量化运算 - 模拟 SIMD"""
    return a + b

# 性能测试
sizes = [100, 1000, 10000, 100000, 1000000]
sisd_times = []
simd_times = []

for size in sizes:
    a_list = list(range(size))
    b_list = list(range(size))
    a_np = np.array(a_list)
    b_np = np.array(b_list)
    
    # SISD
    start = time.perf_counter()
    sisd_add(a_list, b_list)
    sisd_times.append(time.perf_counter() - start)
    
    # SIMD (NumPy uses SIMD instructions internally)
    start = time.perf_counter()
    simd_add(a_np, b_np)
    simd_times.append(time.perf_counter() - start)

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(sizes))
ax1.bar([i - 0.2 for i in x], [t*1000 for t in sisd_times], 0.35, label='SISD (逐个循环)', color='#e74c3c')
ax1.bar([i + 0.2 for i in x], [t*1000 for t in simd_times], 0.35, label='SIMD (NumPy向量化)', color='#27ae60')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{s:,}' for s in sizes], rotation=30)
ax1.set_xlabel('数组大小', fontsize=12)
ax1.set_ylabel('执行时间 (ms)', fontsize=12)
ax1.set_title('SISD vs SIMD 执行时间对比', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# 加速比
speedups = [s/m if m > 0 else 0 for s, m in zip(sisd_times, simd_times)]
ax2.bar(x, speedups, color='#3498db')
ax2.set_xticks(x)
ax2.set_xticklabels([f'{s:,}' for s in sizes], rotation=30)
ax2.set_xlabel('数组大小', fontsize=12)
ax2.set_ylabel('加速比 (Speedup)', fontsize=12)
ax2.set_title('SIMD 相对 SISD 的加速比', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
for i, v in enumerate(speedups):
    ax2.text(i, v + 0.5, f'{v:.1f}x', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 说明：NumPy 内部使用了 CPU 的 SIMD 指令（如 SSE, AVX），")
print("   所以向量化操作比 Python 逐个循环快得多！")

---
## 4. 多核处理器 (Multi-core Processors)

### 4.1 什么是多核处理器？

**多核处理器 (Multi-core Processor)** 是在一块芯片上集成多个独立的处理核心 (core)，每个核心都能独立执行指令。

**核心概念**：
- 每个核心 (core) 是一个完整的处理单元（有自己的 ALU, 控制单元, L1 缓存）
- 多个核心共享 L2/L3 缓存和内存总线
- 操作系统可以将不同任务分配给不同核心 → **真正的并行执行**

### 4.2 多核处理器的优势

| 优势 | 说明 |
|------|------|
| 真正并行 | 多个任务同时在不同核心执行（不是时间片轮转的假并行） |
| 更好的多任务性能 | 同时运行多个应用程序，互不干扰 |
| 功耗效率 | 多个低频核心比单个高频核心更省电（功耗与频率的三次方成正比） |
| 可扩展性 | 可以通过增加核心数来提升性能 |

### 4.3 多核处理器的挑战

| 挑战 | 说明 |
|------|------|
| 软件并行化困难 | 程序必须被设计为多线程才能利用多核 |
| 缓存一致性 (Cache Coherence) | 多核共享数据时需要保证缓存数据一致 |
| 加速比受限 | 阿姆达尔定律：串行部分限制了并行加速比 |
| 编程复杂度 | 多线程编程容易出现竞争条件 (race condition)、死锁 (deadlock) |

In [ ]:
# ============================================================
# 多核处理器架构示意图
# ============================================================

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('四核处理器架构示意图 (Quad-Core Processor)', fontsize=16, fontweight='bold')

# 芯片外框
ax.add_patch(FancyBboxPatch((0.5, 1.5), 13, 7, boxstyle='round,pad=0.2',
                            facecolor='#f8f9fa', edgecolor='#2c3e50', linewidth=3))
ax.text(7, 8.2, 'CPU 芯片 (CPU Die)', ha='center', fontsize=12, fontweight='bold', color='#2c3e50')

# 四个核心
core_positions = [(1.5, 5), (4.5, 5), (7.5, 5), (10.5, 5)]
core_colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for i, ((cx, cy), cc) in enumerate(zip(core_positions, core_colors)):
    # 核心外框
    ax.add_patch(FancyBboxPatch((cx, cy), 2.5, 2.8, boxstyle='round,pad=0.1',
                                facecolor=cc, edgecolor='white', linewidth=2, alpha=0.2))
    ax.text(cx + 1.25, cy + 2.5, f'核心 {i+1} (Core {i+1})', ha='center', fontsize=10, fontweight='bold', color=cc)
    
    # ALU
    ax.add_patch(FancyBboxPatch((cx + 0.15, cy + 1.5), 1.0, 0.7, boxstyle='round,pad=0.05',
                                facecolor=cc, edgecolor='white', alpha=0.7))
    ax.text(cx + 0.65, cy + 1.85, 'ALU', ha='center', fontsize=8, fontweight='bold', color='white')
    
    # CU
    ax.add_patch(FancyBboxPatch((cx + 1.35, cy + 1.5), 1.0, 0.7, boxstyle='round,pad=0.05',
                                facecolor=cc, edgecolor='white', alpha=0.7))
    ax.text(cx + 1.85, cy + 1.85, 'CU', ha='center', fontsize=8, fontweight='bold', color='white')
    
    # L1 Cache
    ax.add_patch(FancyBboxPatch((cx + 0.3, cy + 0.3), 1.9, 0.8, boxstyle='round,pad=0.05',
                                facecolor='#ecf0f1', edgecolor=cc, linewidth=1.5))
    ax.text(cx + 1.25, cy + 0.7, 'L1 Cache', ha='center', fontsize=8, fontweight='bold', color=cc)

# 共享 L2 Cache
ax.add_patch(FancyBboxPatch((1.5, 3.3), 11.5, 1.0, boxstyle='round,pad=0.1',
                            facecolor='#d5dbdb', edgecolor='#2c3e50', linewidth=2))
ax.text(7.25, 3.8, '共享 L2/L3 缓存 (Shared L2/L3 Cache)', ha='center', fontsize=11, fontweight='bold', color='#2c3e50')

# 内存控制器
ax.add_patch(FancyBboxPatch((3.5, 1.8), 7, 0.9, boxstyle='round,pad=0.1',
                            facecolor='#aab7b8', edgecolor='#2c3e50', linewidth=2))
ax.text(7, 2.25, '内存控制器 / 总线接口 (Memory Controller / Bus Interface)', ha='center', fontsize=10, fontweight='bold', color='white')

# 连接线
for cx, cy in core_positions:
    ax.plot([cx + 1.25, cx + 1.25], [cy, 4.3], 'k-', linewidth=1.5, alpha=0.5)

ax.plot([7.25, 7.25], [3.3, 2.7], 'k-', linewidth=2, alpha=0.7)

plt.tight_layout()
plt.show()

---
## 5. GPU 计算基础

### 5.1 GPU vs CPU

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心数 | 少量强大核心 (4-64) | 大量简单核心 (数千) |
| 设计目标 | 低延迟，处理复杂逻辑 | 高吞吐量，处理大量简单计算 |
| 适合任务 | 串行、分支多的任务 | 大规模并行、数据密集型任务 |
| 处理模式 | MIMD | 主要是 SIMD |
| 缓存 | 大缓存 | 小缓存，更多寄存器 |

### 5.2 GPU 计算的应用

- **图形渲染**：GPU 最初的用途，每个像素独立计算
- **深度学习 (Deep Learning)**：神经网络训练需要大量矩阵运算
- **科学模拟**：分子动力学、天气预报、流体力学
- **加密货币挖矿**：大量并行哈希计算
- **视频编解码**：并行处理视频帧的每个块

### 5.3 GPGPU (General-Purpose GPU)

**GPGPU** = General-Purpose computing on Graphics Processing Units

即利用 GPU 进行通用计算（不只是图形处理）。常用框架：
- **CUDA** (NVIDIA 专有)
- **OpenCL** (跨平台开放标准)

In [ ]:
# ============================================================
# CPU vs GPU 架构对比可视化
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# CPU: 少量大核心 + 大缓存
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('CPU 架构', fontsize=14, fontweight='bold')

# CPU 大核心
cpu_core_positions = [(1, 5.5), (3.5, 5.5), (6, 5.5), (8.5, 5.5)]
for i, (x, y) in enumerate(cpu_core_positions):
    ax1.add_patch(FancyBboxPatch((x, y), 1.5, 1.5, boxstyle='round,pad=0.1',
                                facecolor='#3498db', edgecolor='white', linewidth=2))
    ax1.text(x + 0.75, y + 0.75, f'Core\n{i+1}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

# CPU 缓存（大面积）
ax1.add_patch(FancyBboxPatch((1, 3), 9, 2, boxstyle='round,pad=0.1',
                            facecolor='#85c1e9', edgecolor='white', linewidth=2, alpha=0.7))
ax1.text(5.5, 4, '大缓存 (Large Cache)', ha='center', va='center', fontsize=12, fontweight='bold', color='white')

ax1.add_patch(FancyBboxPatch((1, 1), 9, 1.5, boxstyle='round,pad=0.1',
                            facecolor='#aed6f1', edgecolor='white', linewidth=2, alpha=0.5))
ax1.text(5.5, 1.75, '控制逻辑 (Control Logic)', ha='center', va='center', fontsize=11, fontweight='bold', color='#2c3e50')

ax1.text(5.5, 0.3, '少量强大核心 + 大缓存 + 复杂控制', ha='center', fontsize=10, style='italic', color='#2c3e50')

# GPU: 大量小核心
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('GPU 架构', fontsize=14, fontweight='bold')

# GPU 大量小核心
for row in range(8):
    for col in range(16):
        x = 0.5 + col * 0.55
        y = 1.5 + row * 0.65
        ax2.add_patch(plt.Rectangle((x, y), 0.45, 0.55, facecolor='#2ecc71', edgecolor='white', linewidth=0.5, alpha=0.8))

ax2.text(5, 7.5, f'128个 小核心（实际可达数千）', ha='center', fontsize=11, fontweight='bold', color='#27ae60')

# 小缓存
ax2.add_patch(FancyBboxPatch((0.5, 0.5), 8.5, 0.7, boxstyle='round,pad=0.1',
                            facecolor='#82e0aa', edgecolor='white', linewidth=2, alpha=0.5))
ax2.text(4.75, 0.85, '小缓存 (Small Cache)', ha='center', va='center', fontsize=10, fontweight='bold', color='white')

ax2.text(5, 0.1, '大量简单核心 + 小缓存 + 简单控制', ha='center', fontsize=10, style='italic', color='#27ae60')

plt.tight_layout()
plt.show()

---
## 6. 阿姆达尔定律 (Amdahl's Law)

### 6.1 定律内容

**阿姆达尔定律**描述了程序中的**串行部分**对并行加速比的限制。

$$\text{Speedup} = \frac{1}{(1 - P) + \frac{P}{N}}$$

其中：
- $P$ = 可并行化的比例 (0 到 1)
- $N$ = 处理器/核心数量
- $1 - P$ = 必须串行执行的比例

### 6.2 关键启示

- 即使 $N \to \infty$（无限多核心），加速比的上限为 $\frac{1}{1-P}$
- 如果程序有 5% 必须串行 ($P = 0.95$)，最大加速比只有 $\frac{1}{0.05} = 20$，无论用多少核心！
- 如果有 50% 串行 ($P = 0.5$)，最大加速比只有 2！

### 6.3 计算示例

**例题**：一个程序 80% 可以并行化，使用 4 核处理器，求加速比。

$$\text{Speedup} = \frac{1}{(1 - 0.8) + \frac{0.8}{4}} = \frac{1}{0.2 + 0.2} = \frac{1}{0.4} = 2.5$$

即使 80% 可以并行，4核只能达到 2.5 倍加速（不是理想的 4 倍）！

In [ ]:
# ============================================================
# 阿姆达尔定律可视化
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 左图：不同并行比例下的加速比
N = np.arange(1, 65)
parallel_fractions = [0.5, 0.75, 0.9, 0.95, 0.99]
colors_amdahl = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71', '#9b59b6']

for P, c in zip(parallel_fractions, colors_amdahl):
    speedup = 1 / ((1 - P) + P / N)
    max_speedup = 1 / (1 - P)
    ax1.plot(N, speedup, color=c, linewidth=2, label=f'P={P:.0%} (最大={max_speedup:.1f})')
    ax1.axhline(y=max_speedup, color=c, linestyle=':', alpha=0.4)

ax1.plot(N, N, 'k--', alpha=0.3, label='理想线性加速 (P=100%)')
ax1.set_xlabel('处理器数量 N', fontsize=12)
ax1.set_ylabel('加速比 (Speedup)', fontsize=12)
ax1.set_title("阿姆达尔定律 (Amdahl's Law)", fontsize=14, fontweight='bold')
ax1.legend(fontsize=9, loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, 64)
ax1.set_ylim(1, 32)

# 右图：最大加速比 vs 串行比例
serial_fraction = np.linspace(0.01, 0.5, 100)
max_speedup = 1 / serial_fraction

ax2.plot(serial_fraction * 100, max_speedup, 'b-', linewidth=2.5)
ax2.fill_between(serial_fraction * 100, max_speedup, alpha=0.1, color='blue')

# 标注关键点
key_points = [(5, 20), (10, 10), (20, 5), (50, 2)]
for sp, su in key_points:
    ax2.plot(sp, su, 'ro', markersize=8)
    ax2.annotate(f'{sp}% 串行\n最大 {su}x', xy=(sp, su), xytext=(sp+3, su+2),
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))

ax2.set_xlabel('串行部分比例 (%)', fontsize=12)
ax2.set_ylabel('最大加速比 (N→∞)', fontsize=12)
ax2.set_title('串行瓶颈对最大加速比的影响', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 交互式阿姆达尔定律计算器
# ============================================================

def amdahl_calculator(parallel_pct=80, n_cores=4):
    """交互式阿姆达尔定律计算器"""
    P = parallel_pct / 100
    S = 1 - P  # 串行比例
    
    speedup = 1 / (S + P / n_cores)
    max_speedup = 1 / S if S > 0 else float('inf')
    efficiency = speedup / n_cores * 100  # 并行效率
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    # 左图：时间分解
    # 原始
    ax1.barh(['原始 (1核)'], [S], color='#e74c3c', label=f'串行 ({S:.0%})')
    ax1.barh(['原始 (1核)'], [P], left=[S], color='#3498db', label=f'并行 ({P:.0%})')
    
    # 并行后
    ax1.barh([f'并行 ({n_cores}核)'], [S], color='#e74c3c')
    ax1.barh([f'并行 ({n_cores}核)'], [P/n_cores], left=[S], color='#3498db', alpha=0.6)
    
    ax1.set_xlabel('归一化执行时间', fontsize=11)
    ax1.set_title('执行时间分解', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.set_xlim(0, 1.1)
    
    # 右图：加速比 vs 核心数
    cores = np.arange(1, 65)
    speedups = 1 / (S + P / cores)
    ax2.plot(cores, speedups, 'b-', linewidth=2)
    ax2.plot(n_cores, speedup, 'ro', markersize=10, zorder=5)
    ax2.axhline(y=max_speedup, color='red', linestyle='--', alpha=0.5, label=f'最大加速比 = {max_speedup:.1f}')
    ax2.set_xlabel('核心数', fontsize=11)
    ax2.set_ylabel('加速比', fontsize=11)
    ax2.set_title('加速比曲线', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 阿姆达尔定律计算结果：")
    print(f"   可并行比例 P = {P:.0%}")
    print(f"   串行比例 (1-P) = {S:.0%}")
    print(f"   处理器数量 N = {n_cores}")
    print(f"\n   Speedup = 1 / ({S} + {P}/{n_cores})")
    print(f"           = 1 / ({S} + {P/n_cores:.4f})")
    print(f"           = 1 / {S + P/n_cores:.4f}")
    print(f"           = {speedup:.3f}")
    print(f"\n   并行效率 (Efficiency) = {speedup:.3f} / {n_cores} = {efficiency:.1f}%")
    print(f"   理论最大加速比 (N→∞) = 1 / {S} = {max_speedup:.1f}")

widgets.interact(
    amdahl_calculator,
    parallel_pct=widgets.IntSlider(min=0, max=99, step=1, value=80, description='并行比例%:'),
    n_cores=widgets.IntSlider(min=1, max=64, step=1, value=4, description='核心数:')
)

---
## 7. 综合对比与总结

### 处理器性能提升策略总结

| 策略 | 原理 | 优势 | 限制 |
|------|------|------|------|
| 提高时钟频率 | 让每个周期更短 | 简单直接 | 功耗和散热瓶颈 |
| 流水线 (Pipelining) | 指令重叠执行 | 不增加硬件即可加速 | 冒险问题 |
| 超标量 (Superscalar) | 多条流水线并行 | 更高 IPC | 需要复杂调度 |
| 多核 (Multi-core) | 多个独立核心 | 真正并行 | 需要并行软件 |
| SIMD | 单指令操作多数据 | 数据并行高效 | 只适合规则计算 |
| GPU 计算 | 大量简单核心 | 超大规模并行 | 不适合串行逻辑 |

### 考试核心知识点

1. **RISC vs CISC**：能列出至少 4 个区别，并解释各自的优缺点
2. **流水线**：理解5个阶段、加速比公式、三种冒险及解决方案
3. **Flynn 分类**：SISD, SIMD, MISD, MIMD 的定义和例子
4. **多核处理器**：优势和挑战
5. **阿姆达尔定律**：公式计算和含义理解

---
## 8. 练习题 (Practice Exercises)

### 练习 1：RISC vs CISC

**题目**：解释为什么 RISC 处理器更适合流水线技术。列出两个原因。(4分)

<details>
<summary>点击查看答案</summary>

1. RISC 指令长度固定（fixed-length），所以 Fetch 阶段每次取相同大小的数据，流水线各阶段时间一致，不会出现阶段时间不平衡的问题。(2分)

2. RISC 指令简单，大多在一个时钟周期内完成（single-cycle），每个流水线阶段的工作量相等，减少了流水线暂停（stall）的情况。(2分)
</details>

### 练习 2：流水线计算

**题目**：一个 5 阶段流水线处理器执行 100 条指令。

(a) 不使用流水线需要多少个时钟周期？(1分)

(b) 使用流水线需要多少个时钟周期？(1分)

(c) 计算加速比。(2分)

<details>
<summary>点击查看答案</summary>

(a) 100 × 5 = **500** 个周期

(b) 5 + (100 - 1) = **104** 个周期

(c) 加速比 = 500 / 104 = **4.81**（接近理论最大值 5）
</details>

### 练习 3：阿姆达尔定律

**题目**：一个程序的 90% 可以并行化。

(a) 使用 8 核处理器，计算加速比。(3分)

(b) 使用无限多核处理器，最大加速比是多少？(2分)

(c) 解释为什么加倍核心数不一定能加倍性能。(2分)

<details>
<summary>点击查看答案</summary>

(a) Speedup = 1 / (0.1 + 0.9/8) = 1 / (0.1 + 0.1125) = 1 / 0.2125 = **4.71** (3分)

(b) 最大加速比 = 1 / (1 - 0.9) = 1 / 0.1 = **10** (2分)

(c) 因为程序中的串行部分无法被并行化（这里是10%），这部分的执行时间不会因增加核心而减少。根据阿姆达尔定律，串行部分成为瓶颈，限制了整体加速比。增加核心只能减少并行部分的时间，但串行部分保持不变。(2分)
</details>

### 练习 4：Flynn 分类

**题目**：将以下场景分别归类到 SISD、SIMD 或 MIMD：

(a) 一台普通的单核笔记本电脑执行一个计算器程序

(b) GPU 同时对图片的所有像素应用模糊滤镜

(c) 四核 CPU 同时运行浏览器、音乐播放器和文字处理器

<details>
<summary>点击查看答案</summary>

(a) **SISD** - 单指令流，单数据流，逐条执行指令

(b) **SIMD** - 同一条模糊指令同时应用到多个像素（多数据）

(c) **MIMD** - 每个核心执行不同的程序（不同指令），处理不同的数据
</details>

### 练习 5：分支预测

**题目**：解释什么是分支预测(branch prediction)，以及预测错误时会发生什么。(4分)

<details>
<summary>点击查看答案</summary>

分支预测是处理器在条件分支指令（如 if/else）的结果确定之前，**猜测**分支方向的技术。(1分)

处理器根据预测结果，提前将预测路径上的指令加载到流水线中执行。(1分)

如果预测**正确**，流水线继续正常工作，不浪费时间。(1分)

如果预测**错误**，已经进入流水线的指令必须被丢弃（pipeline flush），流水线需要重新加载正确路径上的指令，这会导致若干时钟周期的浪费。(1分)
</details>

In [ ]:
# ============================================================
# 交互式练习：流水线模拟器
# ============================================================

def pipeline_simulator(n_instructions=6, has_hazard=False, hazard_at=2):
    """模拟流水线执行，可选择是否有数据冒险"""
    stages = ['IF', 'ID', 'EX', 'MEM', 'WB']
    colors_s = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
    
    fig, ax = plt.subplots(figsize=(16, max(4, n_instructions * 0.7 + 1)))
    
    max_cycle = 0
    bubble_count = 0
    
    for i in range(n_instructions):
        delay = 0
        if has_hazard and i == hazard_at:
            delay = 2  # 数据冒险导致2个气泡
            bubble_count = 2
        elif has_hazard and i > hazard_at:
            delay = 2
        
        for s in range(5):
            x = i + s + delay
            rect = plt.Rectangle((x, i - 0.35), 0.9, 0.7,
                                facecolor=colors_s[s], edgecolor='white', linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x + 0.45, i, stages[s], ha='center', va='center',
                   fontsize=8, fontweight='bold', color='white')
            max_cycle = max(max_cycle, x + 1)
    
    # 画气泡
    if has_hazard:
        for b in range(2):
            x = hazard_at + b
            y = hazard_at
            rect = plt.Rectangle((x, y - 0.35), 0.9, 0.7,
                                facecolor='#bdc3c7', edgecolor='white', linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x + 0.45, y, 'STALL', ha='center', va='center',
                   fontsize=7, fontweight='bold', color='#7f8c8d')
    
    ax.set_xlim(-0.5, max_cycle + 0.5)
    ax.set_ylim(-0.5, n_instructions + 0.5)
    ax.set_xlabel('时钟周期 (Clock Cycle)', fontsize=12)
    ax.set_ylabel('指令编号', fontsize=12)
    ax.set_yticks(range(n_instructions))
    ax.set_yticklabels([f'I{i+1}' for i in range(n_instructions)])
    ax.set_xticks(range(int(max_cycle) + 1))
    ax.set_xticklabels([str(i+1) for i in range(int(max_cycle) + 1)], fontsize=8)
    
    title = '流水线模拟'
    if has_hazard:
        title += f' (指令 I{hazard_at+1} 处有数据冒险，插入2个气泡)'
    ax.set_title(title, fontsize=13, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    ideal = 5 + n_instructions - 1
    actual = int(max_cycle)
    print(f"总周期数: {actual} (理想无冒险: {ideal})")
    if has_hazard:
        print(f"冒险导致额外的 {actual - ideal} 个周期浪费")

widgets.interact(
    pipeline_simulator,
    n_instructions=widgets.IntSlider(min=3, max=10, value=6, description='指令数:'),
    has_hazard=widgets.Checkbox(value=False, description='启用数据冒险'),
    hazard_at=widgets.IntSlider(min=1, max=8, value=2, description='冒险位置:')
)

---
## 本笔记本要点回顾

1. **RISC** 指令简单、长度固定、寄存器多、适合流水线；**CISC** 指令复杂、长度可变、代码短、功耗高
2. **流水线**通过重叠执行提高吞吐量，加速比公式 $S = \frac{nk}{k+n-1}$，三种冒险及解决方案
3. **分支预测**减少控制冒险的流水线清空代价，2-bit 预测器是常用方案
4. **SIMD** 单指令多数据（GPU核心思想），**MIMD** 多指令多数据（多核CPU）
5. **多核处理器**提供真并行，但需要并行软件支持
6. **阿姆达尔定律** $S = \frac{1}{(1-P)+P/N}$，串行部分是加速比的瓶颈

> **下一节**：02_虚拟机与布尔代数.ipynb